In [1]:
from google.colab import files
uploaded = files.upload()

Saving audio_features.csv to audio_features.csv


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Check if CUDA (GPU) is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the data from your CSV file
data = pd.read_csv("audio_features.csv")
Y_AF = data["Label"]
# Split data into features (X) and labels (y)
X_AF = data.drop(columns=["Link", "Label"])


# Use LabelEncoder to convert string labels to numerical labels
label_encoder = LabelEncoder()
Y_AF = label_encoder.fit_transform(Y_AF)

# Specify the sequence length (adjust as needed)
sequence_length = 20

# Create sequences of the desired length
sequences = []
labels = []
for i in range(len(X_AF) - sequence_length + 1):
    sequence = X_AF.iloc[i:i+sequence_length].values  # Extract a sequence of features
    label = Y_AF[i + sequence_length - 1]  # Use the label of the last item in the sequence
    sequences.append(sequence)
    labels.append(label)

# Convert data to PyTorch tensors and move them to the GPU
sequences = torch.Tensor(sequences).to(device)
labels = torch.LongTensor(labels).to(device)

# Split the data into training and testing sets
X_train_A, X_test_A, y_train_A, y_test_A = train_test_split(sequences, labels, test_size=0.2, random_state=42)

# Create a DataLoader for the training set (optional but useful for mini-batch training)
batch_size = 32  # Adjust as needed
train_dataset = TensorDataset(X_train_A, y_train_A)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Define an LSTM-based model
class EmotionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(EmotionLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # Take the output from the last time step
        return out

# Define the LSTM model hyperparameters
input_size = X_train_A.shape[2]  # Input size based on the number of features in each time step
hidden_size = 64
num_layers = 2  # You can adjust this as needed
num_classes = len(label_encoder.classes_)

# Initialize the model and move it to the GPU
model = EmotionLSTM(input_size, hidden_size, num_layers, num_classes).to(device)

# Define a loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Training loop
num_epochs = 50
for epoch in range(num_epochs):
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# Set the model to evaluation mode
model.eval()

# Make predictions on the test set
with torch.no_grad():
    outputs = model(X_test_A)
    _, predicted = torch.max(outputs, 1)

# Move the predictions to the CPU and convert them to a NumPy array
predicted = predicted.cpu().numpy()

# Calculate accuracy
accuracy = accuracy_score(y_test_A.cpu().numpy(), predicted)
print("Accuracy:", accuracy)

Accuracy: 0.21428571428571427
